# Recommandation d'animes basée sur le contenu

L'objectif de cette partie est de construire un moteur de recommandation basé sur le contenu qui calcule la similarité entre les animes en fonction de leurs genres (`anime.csv`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import html
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import kagglehub
from kagglehub import KaggleDatasetAdapter

animes_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "anime.csv"
)
ratings_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "rating.csv"
)

animes = animes_df[['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']].copy()
ratings = ratings_df[['user_id', 'anime_id', 'rating']].copy()

animes['name'] = animes['name'].astype(str).apply(html.unescape)
animes.head()

/home/yzheng1/Bureau/sys_rec/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama',"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Remplacer les genres manquants par une chaîne vide
animes['genre'] = animes['genre'].fillna('').astype(str)

# Vectorisation TF-IDF sur les genres
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(animes['genre'])
print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)

Dimensions de la matrice TF-IDF : (12294, 654)


In [3]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim[:4, :4]

array([[1.        , 0.04304778, 0.        , 0.        ],
       [0.04304778, 1.        , 0.04968007, 0.        ],
       [0.        , 0.04968007, 1.        , 0.11713654],
       [0.        , 0.        , 0.11713654, 1.        ]])

In [4]:
titles = animes['name']
indices = pd.Series(animes.index, index=animes['name'])

def genre_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:21]
    anime_indices = [i[0] for i in sim_scores]
    return titles.iloc[anime_indices]

In [5]:
genre_recommendations('Hunter x Hunter').head(20)
genre_recommendations('Death Note').head(20)

778                                     Death Note Rewrite
981                                        Mousou Dairinin
144                          Higurashi no Naku Koro ni Kai
1383                         Higurashi no Naku Koro ni Rei
334                              Higurashi no Naku Koro ni
7986                                          Bloody Night
1238                                             Shigofumi
1861                               Himitsu: The Revelation
3829                              Hikari to Mizu no Daphne
38                                                 Monster
833                               Jigoku Shoujo Mitsuganae
6323                               Saint Luminous Jogakuin
315      Ghost in the Shell: Stand Alone Complex - The ...
5382                                             AD Police
471                        Kara no Kyoukai 4: Garan no Dou
834                        Kara no Kyoukai 1: Fukan Fuukei
10880                            Zhongguo Jingqi Xianshe

## Profil d'un utilisateur et recommandation personnalisée

In [6]:
# on retire les notes -1 (animés non notés)
ratings = ratings[ratings['rating'] != -1]

user_id = 5

# les anime_id des animés notés 8 ou plus par l'utilisateur user_id
films_aimes = ratings.loc[(ratings['user_id'] == user_id) & (ratings['rating'] >= 8), 'anime_id']

print(len(films_aimes), "animes aimés")
animes[animes['anime_id'].isin(films_aimes)].head()

45 animes aimés


,anime_id,name,genre,type,episodes,rating,members
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama',"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266
8,15335,Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...,"Action, Comedy, Historical, Parody, Samurai, S...",Movie,1,9.10,72534
9,15417,Gintama': Enchousen,"Action, Comedy, Historical, Parody, Samurai, S...",TV,13,9.11,81109
12,918,Gintama,"Action, Comedy, Historical, Parody, Samurai, S...",TV,201,9.04,336376


In [7]:
# Les numéros de ligne de ces animes dans animes, donc dans tfidf_matrix
lignes = np.flatnonzero(animes['anime_id'].isin(films_aimes))


profil = tfidf_matrix[lignes].mean(axis=0)
profil = np.asarray(profil).reshape(1, -1)
profil.shape


(1, 654)

In [8]:
termes = tf.get_feature_names_out()
poids = pd.Series(profil.ravel(), index=termes)

poids.sort_values(ascending=False).head(5)

comedy     0.101566
shounen    0.099208
drama      0.088480
action     0.076494
sports     0.064749
dtype: float64

In [9]:
def recommander_pour(user_id, notes, n=10):
    notes_user = notes[notes['user_id'] == user_id]
    films_aimes = notes_user.loc[notes_user['rating'] >= 8, 'anime_id']
    lignes = np.flatnonzero(animes['anime_id'].isin(films_aimes))


    if len(lignes) == 0:
        return []


    profil = np.asarray(tfidf_matrix[lignes].mean(axis=0)).reshape(1, -1)
    deja_vus = animes['anime_id'].isin(notes_user['anime_id']).to_numpy()
    #calcul du score de similarité de chaque animé avec le profil de l'user
    scores = linear_kernel(profil, tfidf_matrix).ravel()
    scores[deja_vus] = -1 #exclusion des animés déjà vus
    positions = np.argsort(-scores, kind='stable')[:n]

    return animes['anime_id'].iloc[positions].tolist()




def afficher(anime_ids):
    return animes.set_index('anime_id').loc[list(anime_ids), ['name', 'genre']]

afficher(recommander_pour(5, ratings))

,name,genre
anime_id,,
1186,Battle Athletess Daiundoukai (TV),"Action, Adventure, Comedy, Drama, School, Sci-..."
9078,Rokudenashi Blues 1993,"Action, Comedy, Drama, School, Shounen, Sports"
1670,Bakuten Shoot Beyblade the Movie: Gekitou!! Ta...,"Action, Adventure, Comedy, Sci-Fi, Shounen, Sp..."
10731,Prince of Tennis: Eikokushiki Teikyuu Shiro Ke...,"Action, Comedy, School, Shounen, Sports"
1327,Aoki Densetsu Shoot!,"Action, Comedy, Drama, Romance, School, Shoune..."
31933,JoJo no Kimyou na Bouken: Diamond wa Kudakenai,"Action, Adventure, Comedy, Drama, Shounen, Sup..."
154,Shaman King,"Action, Adventure, Comedy, Drama, Shounen, Sup..."
32935,Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...,"Comedy, Drama, School, Shounen, Sports"
29755,Haikyuu!! Movie 1: Owari to Hajimari,"Comedy, Drama, School, Shounen, Sports"


In [10]:
afficher(recommander_pour(7, ratings))
# Les 30 meilleurs scores de l'utilisateur (son profil vient de la partie 1)
scores_1 = linear_kernel(profil, tfidf_matrix).ravel()
np.sort(scores_1)[::-1][:30].round(3)

array([0.198, 0.195, 0.19 , 0.19 , 0.19 , 0.19 , 0.188, 0.188, 0.185,
       0.175, 0.175, 0.174, 0.174, 0.174, 0.174, 0.174, 0.174, 0.174,
       0.174, 0.173, 0.172, 0.172, 0.169, 0.168, 0.167, 0.167, 0.167,
       0.167, 0.167, 0.167])

## Évaluation et comparaison avec les animes les plus populaires

In [11]:
from sklearn.model_selection import train_test_split

small_ratings = ratings.sample(n=50000, random_state=42)
notes_train, notes_test = train_test_split(small_ratings, test_size=0.2, random_state=42)

# Pour chaque utilisateur, l'ensemble des animes qu'il a aimés (note >= 8) parmi ses notes cachées
aimes_test = notes_test[notes_test['rating'] >= 8].groupby('user_id')['anime_id'].apply(set)
print(len(aimes_test), "utilisateurs ont au moins un anime aimé dans le test")

5427 utilisateurs ont au moins un anime aimé dans le test


In [12]:
def precision_at_k(recos, aimes, k=10):
    """La part des k premières recommandations que l'utilisateur a aimées (aimes est un set)."""
    return len(set(recos[:k]) & aimes) / k


# Les anime_id du train, du plus noté au moins noté
popularite = notes_train['anime_id'].value_counts()

def recommander_populaires(user_id, notes, n=10):
    """Les n films les plus notés que user_id n'a pas encore notés."""
    deja_vus = set(notes.loc[notes['user_id'] == user_id, 'anime_id'])
    return [m for m in popularite.index if m not in deja_vus][:n]


print(precision_at_k([1, 2, 3, 4], {2, 4, 99}, k=4))   # attendu : 0.5
afficher(recommander_populaires(5, notes_train))

0.5


,name,genre
anime_id,,
1535,Death Note,"Mystery, Police, Psychological, Supernatural, ..."
226,Elfen Lied,"Action, Drama, Horror, Psychological, Romance,..."
1575,Code Geass: Hangyaku no Lelouch,"Action, Mecha, Military, School, Sci-Fi, Super..."
6547,Angel Beats!,"Action, Comedy, Drama, School, Supernatural"
11757,Sword Art Online,"Action, Adventure, Fantasy, Game, Romance"
2904,Code Geass: Hangyaku no Lelouch R2,"Action, Drama, Mecha, Military, Sci-Fi, Super ..."
20,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P..."
16498,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power"
121,Fullmetal Alchemist,"Action, Adventure, Comedy, Drama, Fantasy, Mag..."
